# 03 — ARIMA / ARIMAX Baseline
Auto-select ARIMA order, add exogenous covariates (ARIMAX), rolling out-of-sample forecast.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120

## 1. Load features

In [ ]:
train = pd.read_csv('../data/processed/train.csv', index_col=0, parse_dates=True)
val   = pd.read_csv('../data/processed/val.csv',   index_col=0, parse_dates=True)
test  = pd.read_csv('../data/processed/test.csv',  index_col=0, parse_dates=True)

TARGET = 'silver_return'
EXOG   = ['gold_return', 'usd_return', 'copper_return', 'sp500_return']

y_train = train[TARGET].dropna()
y_test  = test[TARGET].dropna()
print(y_train.shape, y_test.shape)

## 2. Order selection via AIC grid search

In [ ]:
best_aic, best_order = np.inf, (1, 0, 1)
results = []

for p in range(0, 5):
    for q in range(0, 5):
        try:
            m = ARIMA(y_train, order=(p, 0, q)).fit()
            results.append({'p': p, 'q': q, 'aic': m.aic, 'bic': m.bic})
            if m.aic < best_aic:
                best_aic, best_order = m.aic, (p, 0, q)
        except Exception:
            pass

results_df = pd.DataFrame(results).sort_values('aic').head(10)
print(f'Best ARIMA order: {best_order}  AIC: {best_aic:.2f}')
results_df

## 3. Fit ARIMA on train, 1-step rolling forecast on test

In [ ]:
def recursive_forecast(y_train, y_test, order, exog_train=None, exog_test=None):
    """Expanding-window 1-step-ahead forecast.
    Trains on all data from the beginning up to t — window grows each step."""
    history = list(y_train)
    preds   = []
    exog_h  = list(exog_train.values) if exog_train is not None else None

    for t in range(len(y_test)):
        exog_f = None
        if exog_h is not None:
            exog_f = [exog_test.iloc[t].values]
        try:
            model = ARIMA(history, order=order,
                          exog=exog_h if exog_h is not None else None).fit()
            fc    = model.forecast(steps=1, exog=exog_f)
            preds.append(float(np.asarray(fc).flat[0]))
        except Exception:
            preds.append(np.nan)
        history.append(float(y_test.iloc[t]))
        if exog_h is not None:
            exog_h.append(exog_test.iloc[t].values)
    return np.array(preds)


def rolling_forecast(y_train, y_test, order, window_size=500,
                     exog_train=None, exog_test=None):
    """Fixed-window 1-step-ahead forecast.
    Trains on the last `window_size` observations only — drops oldest as new data arrives.
    Better for silver because pre/post 2021 squeeze are different regimes."""
    # seed history with the last window_size obs from training
    history = list(y_train[-window_size:])
    preds   = []
    exog_h  = list(exog_train.values[-window_size:]) if exog_train is not None else None

    for t in range(len(y_test)):
        exog_f = None
        if exog_h is not None:
            exog_f = [exog_test.iloc[t].values]
        try:
            model = ARIMA(history, order=order,
                          exog=exog_h if exog_h is not None else None).fit()
            fc    = model.forecast(steps=1, exog=exog_f)
            preds.append(float(np.asarray(fc).flat[0]))
        except Exception:
            preds.append(np.nan)
        # append new observation and drop the oldest to keep window fixed
        history.append(float(y_test.iloc[t]))
        history.pop(0)
        if exog_h is not None:
            exog_h.append(exog_test.iloc[t].values)
            exog_h.pop(0)
    return np.array(preds)


print('Running recursive forecast...')
preds_arima_rec = recursive_forecast(y_train, y_test, best_order)
print('Done')

print('Running rolling forecast (window=500)...')
preds_arima_rol = rolling_forecast(y_train, y_test, best_order, window_size=500)
print('Done')

## 4. ARIMAX — add exogenous variables

In [ ]:
exog_cols = [c for c in EXOG if c in train.columns]
X_train = train[exog_cols].reindex(y_train.index).fillna(0)
X_test  = test[exog_cols].reindex(y_test.index).fillna(0)

print('Running ARIMAX recursive forecast...')
preds_arimax_rec = recursive_forecast(y_train, y_test, best_order,
                                      exog_train=X_train, exog_test=X_test)
print('Done')

print('Running ARIMAX rolling forecast (window=500)...')
preds_arimax_rol = rolling_forecast(y_train, y_test, best_order, window_size=500,
                                    exog_train=X_train, exog_test=X_test)
print('Done')

## 5. Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

actuals = y_test.values

def evaluate(name, y_true, y_pred):
    mask = ~np.isnan(y_pred) & ~np.isnan(y_true)
    if mask.sum() == 0:
        print(f'{name:25s}  No valid predictions')
        return None

    y_t = y_true[mask]
    y_p = y_pred[mask]

    rmse = np.sqrt(mean_squared_error(y_t, y_p))
    mae  = mean_absolute_error(y_t, y_p)

    # plain directional accuracy
    da = np.mean(np.sign(y_t) == np.sign(y_p))

    # MAPE — skip days where actual return is near zero to avoid division explosion
    nonzero = np.abs(y_t) > 1e-6
    mape = np.mean(np.abs((y_t[nonzero] - y_p[nonzero]) / y_t[nonzero])) * 100

    # weighted directional accuracy — big moves contribute more than tiny ones
    correct = np.sign(y_t) == np.sign(y_p)
    weights = np.abs(y_t)
    wda = np.sum(weights * correct) / np.sum(weights)

    print(f'{name:25s}  RMSE={rmse:.6f}  MAE={mae:.6f}  MAPE={mape:.2f}%  DA={da:.3f}  W-DA={wda:.3f}')
    return {'model': name, 'rmse': rmse, 'mae': mae, 'mape': mape, 'dir_acc': da, 'wda': wda}

metrics = []
metrics.append(evaluate('Naive (t-1)',        actuals[1:],         actuals[:-1]))
metrics.append(evaluate('ARIMA recursive',    actuals,             preds_arima_rec))
metrics.append(evaluate('ARIMA rolling',      actuals,             preds_arima_rol))
metrics.append(evaluate('ARIMAX recursive',   actuals,             preds_arimax_rec))
metrics.append(evaluate('ARIMAX rolling',     actuals,             preds_arimax_rol))

metrics_df = pd.DataFrame([m for m in metrics if m is not None])
metrics_df.to_csv('../data/processed/metrics_arima.csv', index=False)
metrics_df

# Save predictions for Diebold-Mariano test in 07_evaluation.ipynb
pd.DataFrame({'actual': actuals, 'predicted': preds_arima_rol}).to_csv(
    '../data/processed/preds_arima_rol.csv', index=False)
pd.DataFrame({'actual': actuals, 'predicted': preds_arimax_rol}).to_csv(
    '../data/processed/preds_arimax_rol.csv', index=False)
print('Prediction arrays saved.')


## 6. Forecast plot

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Top: ARIMA variants
ax = axes[0]
ax.plot(y_test.index, actuals,          label='Actual',           lw=1, color='black')
ax.plot(y_test.index, preds_arima_rec,  label='ARIMA recursive',  lw=1, alpha=0.8)
ax.plot(y_test.index, preds_arima_rol,  label='ARIMA rolling',    lw=1, alpha=0.8, linestyle='--')
ax.set_title('1-step Forecast — Test Set')
ax.set_ylabel('Silver log-return')
ax.legend(fontsize=8)

# Bottom: ARIMAX variants
ax = axes[1]
ax.plot(y_test.index, actuals,           label='Actual',            lw=1, color='black')
ax.plot(y_test.index, preds_arimax_rec,  label='ARIMAX recursive',  lw=1, alpha=0.8)
ax.plot(y_test.index, preds_arimax_rol,  label='ARIMAX rolling',    lw=1, alpha=0.8, linestyle='--')
ax.set_ylabel('Silver log-return')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/processed/arima_forecast_plot.png', dpi=150, bbox_inches='tight')
plt.show()
